# TVTSyn listening validation in VS Code

Review the prepared audio **inside this notebook**. No web server, forwarded port, or model download is needed. Audio bytes are embedded in cell outputs and played by VS Code on your computer, even if Python runs over SSH.

1. Select **`.venv-listening/bin/python`** using **Select Kernel → Python Environments**. If necessary, run `bash inference/setup_listening.sh` in the repository terminal first.
2. Run the setup cells, enter your name, then play and rate each case below. Use headphones at a comfortable fixed volume.
3. Run the final save cell to write `artifacts/listening/listening-results.json`, then share that file.

This reviews prepared file/streaming outputs. **It does not test your live microphone or the Gradio stop/restart UI.** That field remains pending unless you separately complete those checks. Saving a partial review is supported.

If audio players are hidden, trust this repository's notebook and enable the VS Code Jupyter notebook renderers. You do not need to select the older `darkstream` kernel.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inference" / "listening_notebook.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the TVTSyn repository.")
PACK = ROOT / "artifacts/listening"
MODEL = ROOT / "artifacts/TVTSyn"
RESULTS_PATH = PACK / "listening-results.json"
if not (PACK / "manifest.json").is_file():
    raise RuntimeError("Generate the pack first: .venv-listening/bin/python inference/prepare_listening.py --offline")

sys.path.insert(0, str(ROOT / "inference"))
from listening_notebook import load_review, show_case, rate_case, show_summary, save_review
from check_listening import check

manifest, review = load_review(PACK, MODEL, RESULTS_PATH)
print("Interpreter:", sys.executable)
print("Review:", manifest["review_id"][:16])
print("Loaded saved ratings." if RESULTS_PATH.exists() else "New review: all ratings pending.")

## Reviewer
Replace the values below with your name/initials and headphones/browser details. The defaults preserve a saved review. After editing, run the cell.

In [ ]:
reviewer = review["reviewer"]  # Change to e.g. "Waris"
environment = review["environment"]  # E.g. "Headphones; VS Code over SSH"
review["reviewer"] = reviewer
review["environment"] = environment
print("Reviewer:", reviewer or "Not entered yet")

## How to compare and rate

Compare source → each conversion for intelligibility, target voice, onset and final consonants. Compare **PyTorch streaming → ONNX streaming** for clicks, missing sounds and discontinuities. Whole-utterance output is a separate behavior and need not sound identical to streaming.

Stop one player before starting the next. The files already have a common playback gain within each case; the notebook does not normalize each full-file player separately.

In each rating cell, change `result=None` to `result="pass"`, `"fail"`, or `"unsure"`, and add timestamped notes. `None` leaves existing values untouched when rerunning cells. Use `result="pending"` to clear a rating explicitly. Save at the end whenever you want to preserve progress.

## 1. Source → target

Compare all three modes with the source and target reference.

In [ ]:
show_case(PACK, manifest, "forward")

In [ ]:
rate_case(review, "forward", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## 2. Reverse direction

Check conversion in the other speaker direction.

In [ ]:
show_case(PACK, manifest, "reverse")

In [ ]:
rate_case(review, "reverse", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## 3. Short input (80 ms)

Judge clipping, bursts, and draining. This fragment is too short to judge voice identity.

In [ ]:
show_case(PACK, manifest, "short")

In [ ]:
rate_case(review, "short", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## 4. Partial final chunk

Listen for a cut-off or repeated final sound.

In [ ]:
show_case(PACK, manifest, "partial")

In [ ]:
rate_case(review, "partial", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## 5. Silence transitions

Listen for spurious bursts around the start and end of speech.

In [ ]:
show_case(PACK, manifest, "silence")

In [ ]:
rate_case(review, "silence", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## 6. Long stream (30 seconds)

The source repeats intentionally. Check the source at repeat boundaries before attributing clicks to conversion. Pay special attention to **21.2–21.4 s**.

In [ ]:
show_case(PACK, manifest, "long")

### Focused excerpt: 20.8–21.8 seconds

These players use the same saved gain, zoomed into the flagged region. Timestamps in your notes should refer to the full clip.

The long case exceeds the original strict reference tolerance at a few samples (peak difference about 0.00112), while meeting the separately documented extended listening criterion. Identical quantizer code selections were verified; listening is still needed. See `docs/LISTENING_VALIDATION.md` for the numerical details.

In [ ]:
show_case(PACK, manifest, "long", start=20.8, end=21.8)

In [ ]:
rate_case(review, "long", result=None, notes=None)
# Example: result="unsure", notes="Click near 21.3 s in ONNX streaming"

## Physical microphone check — separate from the recorded audio review

Leave this pending if you cannot use the live demo from this setup. The notebook cannot capture your laptop microphone through the remote Python kernel. You can finish and save the recorded-audio review now; the release checker will accurately report the remaining microphone check.

Only mark Pass after testing both backends in the demo: prepare a target, record, stop/drain, record again without preparing, and check a second browser session with another target. See `docs/LISTENING_VALIDATION.md`.

In [ ]:
microphone_result = review["microphone"]["result"]  # pending / pass / fail
microphone_notes = review["microphone"]["notes"]
if microphone_result not in {"pending", "pass", "fail"}:
    raise ValueError("Use pending, pass, or fail")
review["microphone"] = {"result": microphone_result, "notes": microphone_notes}
show_summary(review)

## Save your review

Set `SAVE_REVIEW = True` and run this cell. Pending cases are allowed so you can save progress. The JSON is saved on the repository machine; find it in VS Code's Explorer. Reopen/rerun the setup cell to resume from that file.

Saving does **not** publish anything or mark missing checks as passed. Set the flag back to False before sharing this notebook. Share the JSON instead of committing audio-filled notebook outputs.

In [ ]:
SAVE_REVIEW = False  # Set True after entering your name and ratings.
if SAVE_REVIEW:
    save_review(review, RESULTS_PATH)
else:
    print("Not saved yet. Set SAVE_REVIEW = True to save your ratings.")

## Check what remains

This reports incomplete or failed cases and verifies the model hashes. Pending results on the first run are expected. A completed file review with microphone still pending remains a partial release review.

In [ ]:
remaining = check(review, manifest, MODEL)
if remaining:
    print("Remaining before publication:")
    for item in remaining:
        print("-", item)
else:
    print("All review checks passed. Save the JSON and share it for the publication step.")